<a href="https://colab.research.google.com/github/alvinjanuar05-sudo/Fresh-vs-Spoiled-Food-Detection-CNN/blob/main/Copy_of_Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
print("Upload file kaggle.json lo di bawah ini bro:")
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Sedang mendownload dataset...")
!kaggle datasets download -d swoyam2609/fresh-and-stale-classification
!kaggle datasets download -d maheen00shahid/fresh-and-spoiled-food-image-dataset

print("Sedang mengekstrak file...")
!mkdir -p data_mentah_sayur data_mentah_makanan
!unzip -q fresh-and-stale-classification.zip -d data_mentah_sayur
!unzip -q fresh-and-spoiled-food-image-dataset.zip -d data_mentah_makanan
print("Dataset Kaggle berhasil ditarik!")

Upload file kaggle.json lo di bawah ini bro:


Saving kaggle.json to kaggle.json
Sedang mendownload dataset...
Dataset URL: https://www.kaggle.com/datasets/swoyam2609/fresh-and-stale-classification
License(s): Community Data License Agreement - Permissive - Version 1.0
100% 2.84G/2.84G [00:40<00:00, 74.9MB/s]

Dataset URL: https://www.kaggle.com/datasets/maheen00shahid/fresh-and-spoiled-food-image-dataset
License(s): CC-BY-SA-4.0
100% 570M/570M [00:08<00:00, 66.9MB/s]

Sedang mengekstrak file...
Dataset Kaggle berhasil ditarik!


In [ ]:
import os
import shutil

# 1. Membuat folder tujuan akhir untuk TensorFlow
FOLDER_BAIK = '/content/dataset_gabungan/BAIK'
FOLDER_JELEK = '/content/dataset_gabungan/JELEK'

os.makedirs(FOLDER_BAIK, exist_ok=True)
os.makedirs(FOLDER_JELEK, exist_ok=True)

def sortir_dan_pindahkan(folder_sumber, prefix_dataset):
    count_baik = 0
    count_jelek = 0

    for root, dirs, files in os.walk(folder_sumber):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.jfif')):
                jalur_asli = os.path.join(root, file)
                jalur_huruf_kecil = root.lower()
                ext = os.path.splitext(file)[1]

                # Klasifikasi dan pemindahan langsung ke folder target
                if 'fresh' in jalur_huruf_kecil:
                    nama_baru = f"baik_{prefix_dataset}_{count_baik}{ext}"
                    shutil.move(jalur_asli, os.path.join(FOLDER_BAIK, nama_baru))
                    count_baik += 1
                elif any(x in jalur_huruf_kecil for x in ['stale', 'spoiled', 'rotten']):
                    nama_baru = f"jelek_{prefix_dataset}_{count_jelek}{ext}"
                    shutil.move(jalur_asli, os.path.join(FOLDER_JELEK, nama_baru))
                    count_jelek += 1

# Eksekusi pemindahan data dari mentah ke folder bersih
print("Sedang memproses data sayur...")
sortir_dan_pindahkan('/content/data_mentah_sayur', 'sayur')

print("Sedang memproses data makanan...")
sortir_dan_pindahkan('/content/data_mentah_makanan', 'makan')

print("\nHASIL AKHIR STRUKTUR DATASET BARU:")
print(f"Total file asli di folder BAIK  : {len(os.listdir(FOLDER_BAIK))} gambar")
print(f"Total file asli di folder JELEK : {len(os.listdir(FOLDER_JELEK))} gambar")

Sedang memproses data sayur...
Sedang memproses data makanan...

HASIL AKHIR STRUKTUR DATASET BARU:
Total file asli di folder BAIK  : 15737 gambar
Total file asli di folder JELEK : 17335 gambar


In [ ]:
import tensorflow as tf

PATH_DATASET = "/content/dataset_gabungan"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# 1. Load Dataset Training (80 persen data)
train_ds = tf.keras.utils.image_dataset_from_directory(
    PATH_DATASET,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

# 2. Load Dataset Validasi (20 persen data)
val_ds = tf.keras.utils.image_dataset_from_directory(
    PATH_DATASET,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

# 3. Optimasi performa memori RAM
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

print("Proses load dataset selesai. Perhatikan jumlah total gambar yang terbaca di atas.")

Found 33072 files belonging to 2 classes.
Using 26458 files for training.
Found 33072 files belonging to 2 classes.
Using 6614 files for validation.
Proses load dataset selesai. Perhatikan jumlah total gambar yang terbaca di atas.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def buat_model_sakti():
    # Lapisan augmentasi otomatis di memori (menggantikan script kloning fisik)
    data_augmentation = tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.1),
    ], name="data_augmentation_layer")

    # Load Base Model EfficientNetB3 tanpa kacamata hitam (tanpa layer rescaling manual)
    base_model = tf.keras.applications.EfficientNetB3(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    # Menyusun struktur akhir jaringan AI
    model = tf.keras.Sequential([
        layers.InputLayer(input_shape=(224, 224, 3)),
        data_augmentation,
        base_model,
        layers.GlobalAveragePooling2D(name="global_average_pooling2d"),
        layers.Dropout(0.2, name="dropout_layer"),
        layers.Dense(2, activation='softmax', name='output_layer')
    ])

    return model

# Panggil fungsi dan compile model
model = buat_model_sakti()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Arsitektur Model EfficientNetB3 siap digunakan.")

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Arsitektur Model EfficientNetB3 siap digunakan.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [ ]:
EPOCHS = 12

print("Memulai pertempuran training EfficientNetB3...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

print("\nTraining Tahap 1 Selesai!")

Memulai pertempuran training EfficientNetB3...
Epoch 1/12
827/827 ━━━━━━━━━━━━━━━━━━━━ 179s 192ms/step - accuracy: 0.8126 - loss: 0.4273 - val_accuracy: 0.8783 - val_loss: 0.3205
Epoch 2/12
827/827 ━━━━━━━━━━━━━━━━━━━━ 206s 202ms/step - accuracy: 0.8820 - loss: 0.3029 - val_accuracy: 0.8999 - val_loss: 0.2720
Epoch 3/12
827/827 ━━━━━━━━━━━━━━━━━━━━ 210s 212ms/step - accuracy: 0.8974 - loss: 0.2690 - val_accuracy: 0.9132 - val_loss: 0.2460
Epoch 4/12
827/827 ━━━━━━━━━━━━━━━━━━━━ 156s 189ms/step - accuracy: 0.9059 - loss: 0.2474 - val_accuracy: 0.9179 - val_loss: 0.2322
Epoch 5/12
827/827 ━━━━━━━━━━━━━━━━━━━━ 167s 201ms/step - accuracy: 0.9094 - loss: 0.2383 - val_accuracy: 0.9226 - val_loss: 0.2204
Epoch 6/12
827/827 ━━━━━━━━━━━━━━━━━━━━ 156s 188ms/step - accuracy: 0.9146 - loss: 0.2290 - val_accuracy: 0.9265 - val_loss: 0.2131
Epoch 7/12
827/827 ━━━━━━━━━━━━━━━━━━━━ 166s 201ms/step - accuracy: 0.9167 - loss: 0.2236 - val_accuracy: 0.9274 - val_loss: 0.2068
Epoch 8/12
827/827 ━━━━━━━━━━

In [ ]:
# 1. Mencari dan membuka gembok base model EfficientNetB3 di dalam jaringan
for layer in model.layers:
    if 'efficientnet' in layer.name.lower():
        layer.trainable = True
        print(f"Gembok arsitektur {layer.name} berhasil dibuka untuk Fine-Tuning.")

# 2. Compile ulang model dengan Learning Rate yang jauh lebih kecil (10x lebih kecil)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Lanjutkan training tahap dua sebanyak 8 epoch tambahan
FINE_TUNE_EPOCHS = 8
print("\nMemulai proses Fine-Tuning massal...")

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS
)

print("\nFine-Tuning Selesai!")

Gembok arsitektur efficientnetb3 berhasil dibuka untuk Fine-Tuning.

Memulai proses Fine-Tuning massal...
Epoch 1/8
827/827 ━━━━━━━━━━━━━━━━━━━━ 666s 727ms/step - accuracy: 0.8882 - loss: 0.2691 - val_accuracy: 0.9368 - val_loss: 0.1603
Epoch 2/8
827/827 ━━━━━━━━━━━━━━━━━━━━ 596s 721ms/step - accuracy: 0.9348 - loss: 0.1642 - val_accuracy: 0.9490 - val_loss: 0.1218
Epoch 3/8
827/827 ━━━━━━━━━━━━━━━━━━━━ 606s 733ms/step - accuracy: 0.9474 - loss: 0.1298 - val_accuracy: 0.9569 - val_loss: 0.1033
Epoch 4/8
827/827 ━━━━━━━━━━━━━━━━━━━━ 622s 752ms/step - accuracy: 0.9581 - loss: 0.1040 - val_accuracy: 0.9645 - val_loss: 0.0893
Epoch 5/8
827/827 ━━━━━━━━━━━━━━━━━━━━ 594s 719ms/step - accuracy: 0.9648 - loss: 0.0893 - val_accuracy: 0.9660 - val_loss: 0.0820
Epoch 6/8
827/827 ━━━━━━━━━━━━━━━━━━━━ 595s 720ms/step - accuracy: 0.9690 - loss: 0.0782 - val_accuracy: 0.9686 - val_loss: 0.0807
Epoch 7/8
827/827 ━━━━━━━━━━━━━━━━━━━━ 605s 731ms/step - accuracy: 0.9712 - loss: 0.0726 - val_accuracy: 0.9

In [ ]:
# Menentukan nama file penyimpanan model
NAMA_FILE_MODEL = '/content/model_efficientnet_sakti.keras'

# Proses penyimpanan model ke lokal Colab
model.save(NAMA_FILE_MODEL)

print("PROSES SELESAI TOTAL!")
print(f"File model berhasil disimpan dengan nama: {NAMA_FILE_MODEL}")
print("Silakan buka panel file di sebelah kiri Colab.")

PROSES SELESAI TOTAL!
File model berhasil disimpan dengan nama: /content/model_efficientnet_sakti.keras
Silakan buka panel file di sebelah kiri Colab.
